# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL:
`https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json`

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata
print(f"{getattr(metadata, 'name', None)}: {getattr(metadata, 'description', None)}")

## 2. Data Overview
Review available record sets, fields, and their IDs.

We print the list of record sets, and for each record set, show their fields and columns using their `@id`s.

- If you do not know the record set `@id`, you can list all available ones as below.

In [ ]:
# List record sets and their properties using `@id` references
record_sets = dataset.record_sets
print(f"Number of record sets: {len(record_sets)}\n")
record_set_ids = []
for rs in record_sets:
    print(f"Record Set name: {getattr(rs, 'name', None)}  |  @id: {rs.id}")
    record_set_ids.append(rs.id)
    print("  Fields (by @id):")
    for field in rs.fields:
        print(f"    - {field.id} (name: {getattr(field, 'name', None)})")
    print("  Columns (by @id):")
    for column in getattr(rs, 'columns', []):
        print(f"    - {column.id} (title: {getattr(column, 'name', None)})")
    print()

## 3. Data Extraction
Load data from each record set into a DataFrame for analysis. All references to record sets and fields are via their `@id`s.

In [ ]:
# Load records from each record set into Pandas DataFrames
dataframes = {}
for record_set_id in record_set_ids:
    records = list(dataset.records(record_set=record_set_id))
    # If records are generators of None (empty), skip
    if records and isinstance(records[0], dict):
        df = pd.DataFrame(records)
        dataframes[record_set_id] = df

print("DataFrames loaded for the following record sets (@id):")
for rid, df in dataframes.items():
    print(f"  {rid}: shape={df.shape}")
    print(f"    Columns (@id): {df.columns.tolist()}")

# Select a main record set for demonstration
selected_record_set_id = list(dataframes.keys())[0] if dataframes else None
if selected_record_set_id:
    print(dataframes[selected_record_set_id].head())

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data by group.

In [ ]:
# ---- Choose numeric and group fields by @id ----
import numpy as np

if selected_record_set_id is not None:
    df = dataframes[selected_record_set_id]
    # Try to guess numeric fields
    numeric_candidate = None
    for col in df.columns:
        if np.issubdtype(df[col].dtype, np.number):
            numeric_candidate = col
            break
    if not numeric_candidate:
        # Try 'coverage' or 'abundance' in col name
        for col in df.columns:
            if 'abundance' in col.lower() or 'coverage' in col.lower():
                numeric_candidate = col
                break
    if not numeric_candidate:
        # Try to convert all columns to numeric, keep first convertible
        for col in df.columns:
            try:
                df[col] = pd.to_numeric(df[col])
                if np.issubdtype(df[col].dtype, np.number):
                    numeric_candidate = col
                    break
            except Exception:
                continue

    print(f"Selected numeric field for filtering: {numeric_candidate}")

    if numeric_candidate:
        # Drop NA for statistics
        df = df.dropna(subset=[numeric_candidate])
        # Set a threshold at mean for demo
        threshold = df[numeric_candidate].mean() if len(df) > 0 else 0
        filtered_df = df[df[numeric_candidate] > threshold]
        print(f"Filtered records with {numeric_candidate} > mean:")
        print(filtered_df[[numeric_candidate]].head())

        # Normalization
        filtered_df.loc[:, f"{numeric_candidate}_normalized"] = (
            (filtered_df[numeric_candidate] - filtered_df[numeric_candidate].mean()) /
            (filtered_df[numeric_candidate].std() if filtered_df[numeric_candidate].std() > 0 else 1)
        )
        print(f"Normalized {numeric_candidate} for filtered records:")
        print(filtered_df[[numeric_candidate, f"{numeric_candidate}_normalized"]].head())

        # Try grouping by a likely categorical field
        group_field = None
        category_priorities = ['group', 'category', 'condition', 'sample', 'id', 'type', 'experiment']
        for c in df.columns:
            cname = c.lower()
            if c != numeric_candidate and any(x in cname for x in category_priorities):
                group_field = c
                break
        if group_field:
            print(f"\nGrouping by field: {group_field}...\n")
            grouped_df = filtered_df.groupby(group_field).mean(numeric_only=True)
            print(grouped_df[[numeric_candidate]].head())
        else:
            print("No suitable grouping field found by heuristic.")
    else:
        print("No numeric field found for demonstration.")
else:
    print("No data available for EDA.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

In [ ]:
import matplotlib.pyplot as plt

if selected_record_set_id is not None and numeric_candidate:
    plt.figure(figsize=(8, 4))
    plt.hist(df[numeric_candidate].dropna(), bins=20, color='skyblue', edgecolor='black')
    plt.xlabel(numeric_candidate)
    plt.ylabel('Frequency')
    plt.title(f'Distribution of {numeric_candidate}')
    plt.show()
else:
    print("No numeric field available for visualization.")

## 6. Conclusion
In this notebook, we've demonstrated:
- How to load Croissant metadata and record sets using `mlcroissant`
- How to explore record set structure and extract data using only `@id` references
- Basic filtering, normalization, and grouping of data for exploratory analysis
- Visualization of a selected numeric field

Next steps could include more advanced feature engineering, detailed statistical analysis, or downstream modeling with this biological dataset.